In [1]:
import os
from dotenv import load_dotenv

import textwrap


def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv(r"C:\Users\arunk\OneDrive\Documents\GenAI\LLM\openai_key.env")

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

API key loaded successfully.


In [2]:
from openai import OpenAI

client = OpenAI(api_key=api_key)
pretty_print("OpenAI client ready.")

MODEL  = "gpt-5-nano"

OpenAI client ready.


In [3]:
response = client.responses.create(
    model=MODEL,
    input="What is 1247 * 83 + 19 / 3.7? Give me answer in one line (max 10 words), the exact number, upto 2 decimal places.",
    reasoning={"effort": "minimal"},   
    text={"verbosity": "low"}
)
print("Model says:", response.output_text)

# What Python actually computes
import math
actual = 1247 * 83 + 19 / 3.7
print(f"Actual answer: {actual}")

Model says: 103,601.00
Actual answer: 103506.13513513513


Tool Calling

In [4]:
add_tool = {
    "type": "function",
    "name": "add",
    "description": "Add two numbers together and return the sum.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}


# Ask the model to add — but give it the tool
response = client.responses.create(
    model=MODEL,
    instructions="Use the add tool for any math. Never compute math yourself.",
    input="What is 7 + 12?",
    tools=[add_tool],
)

# Let's inspect what came back
print("Output items from the model:")
print("-" * 40)
for item in response.output:
    print(f"  Type: {item.type}")
    if item.type == "function_call":
        print(f"  Function name: {item.name}")
        print(f"  Arguments: {item.arguments}")
        print(f"  Call ID: {item.call_id}")
pretty_print(" Response: " + response.output_text)

Output items from the model:
----------------------------------------
  Type: reasoning
  Type: function_call
  Function name: add
  Arguments: {"a":7,"b":12}
  Call ID: call_6MKxTlSIfd5LiznXoQ8UstM6
 Response:


In [8]:
import json

def add(a, b):
    return a + b

function_call = [item for item in response.output if item.type == "function_call"]

for function in function_call:
    if function.name == "add":
        args =  json.loads(function.arguments)
        result = add(**args)
        print(f"Result of {args['a']} + {args['b']} = {result}")

Result of 7 + 12 = 19


In [10]:
final = client.responses.create(
    model=MODEL,
    instructions="Always use the add tool for math. Never compute yourself.",
	previous_response_id=response.id,
    input=[{
        "type": "function_call_output",       
        "call_id": function.call_id,    
        "output": str(result),
    }],
    tools=[add_tool],
)

In [11]:
print(final.output_text)

19


Multiple Tools

In [13]:
add_tool = {
    "type": "function",
    "name": "add",
    "description": "Add two numbers together and return the sum.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

sub_tool = {
    "type": "function",
    "name": "subtract",
    "description": "Subtract two numbers and return the difference.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

mul_tool = {
    "type": "function",
    "name": "multiply",
    "description": "Multiply two numbers together and return the product.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}


div_tool = {
    "type": "function",
    "name": "divide",
    "description": "Divide two numbers and return the answer.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "Numerator (dividend)"},
            "b": {"type": "number", "description": "Denominator (divisor)"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    if b == 0:
        return "Error: Division by zero"
    return a / b

DISPATCH = {
    "add": add,
    "subtract": subtract,
    "multiply": multiply,
    "divide": divide,
}

TOOLS = [add_tool, sub_tool, mul_tool, div_tool]

In [14]:
DEV_POLICY = "Use the tools for any math. Never compute math yourself."

def answer_with_math_tools_verbose(user_question: str) -> str:
    print("══════════════════════════════════════════════════════")
    print("START: User question")
    print("  ", user_question)
    print("══════════════════════════════════════════════════════")

    # Round 1: seed conversation with developer policy + user question
    resp = client.responses.create(
        model=MODEL,
        input=[
            {"role": "developer", "content": DEV_POLICY},
            {"role": "user", "content": user_question},
        ],
        tools=TOOLS
    )

    round_num = 1

    while True:
        print(f"\n══════════════════════════════════════════════════════")
        print(f"ROUND {round_num}: Model response items")
        print("══════════════════════════════════════════════════════")

        # Show every output item type
        for i, item in enumerate(resp.output):
            print(f"[{i}] type = {item.type}")

            if item.type == "function_call":
                print(f"    name     = {item.name}")
                print(f"    call_id  = {item.call_id}")
                print(f"    arguments(raw JSON string) = {item.arguments}")

            elif item.type == "message":
                # Some SDKs represent assistant text as message items
                # output_text is the easiest way to get the final combined text.
                try:
                    print(f"    content = {item.content}")
                except Exception:
                    pass

        calls = [item for item in resp.output if item.type == "function_call"]

        # If no tool calls, we’re done; return final user-facing text
        if not calls:
            print("\n✅ No function calls. Final assistant text:")
            print(resp.output_text)
            return resp.output_text

        print("\n→ Model requested tool calls:")
        for call in calls:
            print(f"  - {call.name}({call.arguments})  [call_id={call.call_id}]")

        # Execute all calls and prepare tool outputs
        tool_outputs = []
        print("\n→ Executing tools locally (your server/app):")
        for call in calls:
            fn = DISPATCH.get(call.name)
            args = json.loads(call.arguments)

            try:
                result = fn(**args)
                payload = {"ok": True, "result": result}
                print(f"  ✓ {call.name}(**{args}) -> {result}")
            except Exception as e:
                payload = {"ok": False, "error": str(e)}
                print(f"  ✗ {call.name}(**{args}) -> ERROR: {e}")

            tool_outputs.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                # output is a string; keep it JSON for readability + structure
                "output": json.dumps(payload),
            })

        print("\n→ Sending tool outputs back to the model:")
        for out in tool_outputs:
            pretty_print(out)

        # Continue conversation, ONLY sending tool outputs (chained with previous_response_id)
        resp = client.responses.create(
            model=MODEL,
            previous_response_id=resp.id,
            input=tool_outputs,
            tools=TOOLS,
            reasoning={"effort": "minimal"},
        )

        round_num += 1

# Example
final_text = answer_with_math_tools_verbose("What is 1247 * 83 + 19 / 3.7?")

══════════════════════════════════════════════════════
START: User question
   What is 1247 * 83 + 19 / 3.7?
══════════════════════════════════════════════════════

══════════════════════════════════════════════════════
ROUND 1: Model response items
══════════════════════════════════════════════════════
[0] type = reasoning
[1] type = function_call
    name     = multiply
    call_id  = call_WqDnS9quyjjSXubbpGBn09Mx
    arguments(raw JSON string) = {"a":1247,"b":83}
[2] type = function_call
    name     = divide
    call_id  = call_858BHaqA5rJ7boc3uhhFUeBV
    arguments(raw JSON string) = {"a":19,"b":3.7}

→ Model requested tool calls:
  - multiply({"a":1247,"b":83})  [call_id=call_WqDnS9quyjjSXubbpGBn09Mx]
  - divide({"a":19,"b":3.7})  [call_id=call_858BHaqA5rJ7boc3uhhFUeBV]

→ Executing tools locally (your server/app):
  ✓ multiply(**{'a': 1247, 'b': 83}) -> 103501
  ✓ divide(**{'a': 19, 'b': 3.7}) -> 5.135135135135135

→ Sending tool outputs back to the model:
{'type': 'function_cal

create a tool of weather API

In [57]:
weather_tool = {
    "type": "function",
    "name": "get_weather",
    "description": "Get the current temperature for a given city.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "City name, e.g. 'Tel Aviv', 'London'"
            },
        },
        "required": ["location"],
        "additionalProperties": False,
    },
    "strict": True,
}

In [58]:
response = client.responses.create(
    model=MODEL,
    input="What's the weather in Paris right now?",
    tools=[weather_tool],
)

# Let's inspect what came back
print("Output items from the model:")
print("-" * 40)
for item in response.output:
    print(f"  Type: {item.type}")
    if item.type == "function_call":
        print(f"  Function name: {item.name}")
        print(f"  Arguments: {item.arguments}")
        print(f"  Call ID: {item.call_id}")
pretty_print(" Response: " + response.output_text)

Output items from the model:
----------------------------------------
  Type: reasoning
  Type: function_call
  Function name: get_weather
  Arguments: {"location":"Paris"}
  Call ID: call_yd3DtkcgIXFyTuOMsMOdRAVz
 Response:


In [59]:
load_dotenv('C:\\Users\\arunk\\OneDrive\\Documents\\GenAI - Scaler\\LLM\\open_weather_map_key.env')

OpenWeatherMapAPI_KEY = os.getenv("OPEN_WEATHER_MAP_API_KEY")

if not OpenWeatherMapAPI_KEY:
    raise ValueError(
        "OPEN_WEATHER_MAP_API_KEY not found! "
        "Make sure you have a .env file with: OPEN_WEATHER_MAP_API_KEY=..."
    )

pretty_print("API key loaded successfully.")

API key loaded successfully.


In [60]:
import requests


def get_weather(location):
    """
    Fetches and displays current weather data for a given city.
    """
    # Base URL for the OpenWeatherMap API
    base_url = "https://api.openweathermap.org/data/2.5/weather"
    
    # Parameters for the API request (city name, API key, and units in metric)
    params = {
        "q": location,
        "appid": OpenWeatherMapAPI_KEY,
        "units": "metric" # Use "imperial" for Fahrenheit
    }
    
    try:
        # Send a GET request to the API
        response = requests.get(base_url, params=params)
        
        # Raise an exception for bad status codes (e.g., 404 Not Found)
        response.raise_for_status()
        
        # Convert the JSON response into a Python dictionary
        weather_data = response.json()
        
        # Check if the city was found
        if weather_data.get("cod") != 200:
            print(f"Error: {weather_data.get('message', 'City not found or unknown error')}")
            return

        main_data = weather_data['main']
        weather_info = weather_data['weather'][0]
        
        temperature = main_data['temp']
        humidity = main_data['humidity']
        description = weather_info['description']
        wind_speed = weather_data['wind']['speed']

        # Display the data
        #print(f"\nWeather in {location.title()}:")
        #print(f"Temperature: {temperature}°C")
        #print(f"Humidity: {humidity}%")
        #print(f"Wind Speed: {wind_speed} m/s")
        #print(f"Description: {description.title()}")

        return temperature

    except requests.exceptions.RequestException as e:
        print(f"Network or request error: {e}")
    except json.JSONDecodeError:
        print("Error decoding JSON response from the API.")

In [53]:
get_weather("Paris")


9.89

In [66]:

new_input = []

for item in response.output:
    if item.type == "function_call":
        args = json.loads(item.arguments)
        result = get_weather(**args)
        print(f"  → {item.name}({args}) = {result}")
        
        payload = {"ok": True, "result": result}  
        new_input.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": json.dumps(payload),
        })


  → get_weather({'location': 'Paris'}) = 9.88


In [67]:
final = client.responses.create(
    model=MODEL,
    input=new_input,
    tools=[weather_tool],
    previous_response_id=response.id,
)
print(f"\nFinal answer:\n{final.output_text}")


Final answer:
Current temperature in Paris: about 9.9°C (roughly 50°F).

Would you like more details (conditions, humidity, wind) or a Fahrenheit conversion?
